# Example 10.3 / 3: Cover Shifts with Schedule Columns

Companion notebook for **10.3 Columnwise Formulations** in the AMPL lab Documentation, section "Cover Shifts with Schedule Columns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Cover Shifts with Schedule Columns

In a shift-scheduling model each variable is a weekly schedule, and the data list the shifts each
schedule works in `SHIFT_LIST[j]`. `Work[j]` people follow schedule `j`, and every shift needs at
least `required[i]` people:

$$\min \sum_j rate_j\,Work_j \quad\text{subject to}\quad \sum_{j:\, i \in SHIFT\_LIST_j} Work_j \ge required_i,\; Work_j \ge 0.$$

By rows, each constraint must search the schedules that contain its shift. By columns, a schedule
puts a coefficient of 1 into the row of every shift it works. The keyword `cover` is shorthand for
`coeff ... 1`, so `cover {i in SHIFT_LIST[j]} Shift_Needs[i]` and
`coeff {i in SHIFT_LIST[j]} Shift_Needs[i] 1` give the same rows:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SHIFTS;  # shifts to staff
param Nsched integer > 0;  # number of allowed schedules
set SCHEDS = 1..Nsched;  # schedule numbers
set SHIFT_LIST {SCHEDS} within SHIFTS;  # shifts worked on each schedule
param rate {SCHEDS} >= 0;  # cost per person on each schedule
param required {SHIFTS} >= 0;  # people needed on each shift

minimize Total_Cost;  # terms come from the Work columns
subject to Shift_Needs {i in SHIFTS}: to_come >= required[i];  # staff each shift

var Work {j in SCHEDS} >= 0,  # people assigned to schedule j
    obj Total_Cost rate[j],
    cover {i in SHIFT_LIST[j]} Shift_Needs[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SHIFTS"] = ["Mon", "Tue", "Wed"]
ampl.param["Nsched"] = 3
ampl.set["SHIFT_LIST"][1] = ["Mon", "Tue"]
ampl.set["SHIFT_LIST"][2] = ["Tue", "Wed"]
ampl.set["SHIFT_LIST"][3] = ["Mon", "Wed"]
ampl.param["rate"] = {1: 1, 2: 1, 3: 1}
ampl.param["required"] = {"Mon": 4, "Tue": 5, "Wed": 4}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
print(ampl.var["Work"].to_dict())

HiGHS 1.14.0: optimal solution; objective 6.5
3 simplex iterations
0 barrier iterations
solved 6.5
{1: 2.5, 2: 2.5, 3: 1.5}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SHIFTS := Mon Tue Wed;
param Nsched := 3;
set SHIFT_LIST[1] := Mon Tue;
set SHIFT_LIST[2] := Tue Wed;
set SHIFT_LIST[3] := Mon Wed;
param rate default 1;
param required := Mon 4  Tue 5  Wed 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Work;
expand Shift_Needs;

HiGHS 1.14.0: optimal solution; objective 6.5
3 simplex iterations
0 barrier iterations
Work [*] :=
1  2.5
2  2.5
3  1.5
;

subject to Shift_Needs['Mon']:
	Work[1] + Work[3] >= 4;

subject to Shift_Needs['Tue']:
	Work[1] + Work[2] >= 5;

subject to Shift_Needs['Wed']:
	Work[2] + Work[3] >= 4;

